<div style="display: flex; align-items: center;">
    <h1>Tutorial on hybrid crop modelling with diffWOFOST</h1>
    <img src="https://raw.githubusercontent.com/WUR-AI/diffWOFOST/refs/heads/main/docs/logo/diffwofost.png" width="150" style="margin-left: 20px;">
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ronvree/diffwofost-tutorial/blob/master/hybrid_stress_correction.ipynb)

In this tutorial we will build a **hybrid crop model** by
plugging a small neural network into WOFOST72 and training
it end-to-end. The notebook
uses the [diffwofost](https://github.com/WUR-AI/diffWOFOST) Python package and a public
field-trial dataset.

By the end you will have:

- explored a real field-trial dataset (potato, two Dutch sites, 174 plot-years);
- replaced WOFOST's standard evapotranspiration block with a learnable stress
  factor (`RFTRA`) produced by a neural network;
- compared the hybrid against an uncalibrated WOFOST72_PP reference *and*
  against a pure-ML LSTM with no physics in the loop;
- seen a teaser of what else differentiability buys you (parameter sensitivities
  for free).

> **Citations.** The field data is from Ten Den et al. (2024), Harvard
> Dataverse [doi:10.7910/DVN/1LC6W7](https://doi.org/10.7910/DVN/1LC6W7),
> CC BY-NC-SA 4.0. The diffwofost engine and this tutorial are
> EUPL-1.1. Pretrained models in this notebook are derivative works of
> the field data and are therefore also released under CC BY-NC-SA 4.0.


## 1. The big picture: why hybrid modelling?

Mechanistic crop models like **WOFOST** encode decades of agronomic knowledge as
ODEs (Ordinary Differential Equation): photosynthesis, partitioning, water balance, phenology. They are
*interpretable*, *physically grounded*, and *generalise* to weather and
management regimes outside their calibration data — but they have to make
simplifying assumptions, and the assumptions sometimes break.

Pure **data-driven** models (an LSTM that maps weather + soil → yield) can
absorb whatever structure is in the data, but they have **no inductive bias**:
nothing tells them carbon is conserved, that leaves grow before tubers, or that
photosynthesis caps at light saturation. They overfit small datasets badly and
extrapolate poorly outside the training distribution.

**Hybrid models** keep the physics where it's well-understood and let a neural
network fill in the parts that are poorly modelled. Concretely, this tutorial:

| Component | Source |
|-----------|--------|
| Phenology (DVS, TSUM) | physical (WOFOST) |
| Carbon partitioning to leaves/stems/tubers | physical (WOFOST) |
| Soil water balance (Potetial Production variant) | physical (WOFOST) |
| **Stress reduction factor (`RFTRA`)** | **learned NN** |

In standard WOFOST, `RFTRA` represents a transpiration reduction factor:
a scalar that reduces gross assimilation under water-limited conditions.
Strictly speaking, it is intended to capture water stress only.

In this tutorial, however, we deliberately use `RFTRA` more broadly as a
generic stress gate on assimilation. The `_PP` (potential production) variant
of WOFOST does not model nutrient stress, and therefore systematically
overpredicts growth on nitrogen-limited plots. Instead of implementing the full
water- and nitrogen-limited production variants, we train a small neural network
to infer an effective daily stress factor directly from weather and treatment
context.

The physics still governs phenology, carbon allocation, and crop growth
dynamics; the neural network only modulates how much carbon is fixed each day.

**Disclaimer** This is admittedly a modelling shortcut — we are “misusing” `RFTRA` beyond its
original physiological interpretation — but it is defensible because RFTRA
acts multiplicatively on gross carbon assimilation. From the model’s perspective, both drought stress and nitrogen stress ultimately reduce canopy assimilation, even if the underlying physiological mechanisms differ.

> ❓ **Discuss with a neighbour**
>
> What other parts of a crop model might
> be good candidates for a hybrid replacement? What are the trade-offs of
> replacing more physics with NN vs. less?
>


## 2. Run me first (Colab setup)

Run the collapsed setup cells below (click **Show code** if you would like to see the code (not required)). They install diffwofost, and download the data

In [ ]:
#@title Install diffwofost { display-mode: "form" }
!pip install -q "diffwofost @ git+https://github.com/WUR-AI/diffWOFOST@0a4d4a3b6682"

In [ ]:
#@title Imports and global settings { display-mode: "form" }
import copy
import warnings
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import openpyxl
from openpyxl import Workbook
from pcse.base import ParameterProvider
from pcse.input import (
    ExcelWeatherDataProvider,
    YAMLAgroManagementReader,
    YAMLCropDataProvider,
    WOFOST72SiteDataProvider,
)
from pcse.util import DummySoilDataProvider
from pcse.traitlets import Instance

# diffwofost @ main (commit 0a4d4a3) — physics engine, Configuration
#   with CROP_COMPONENTS support (post-v0.4.0), classic_waterbalance.
# StressNN and NNStressFactor are NOT on PyPI yet; defined in §5.1.
from diffwofost.physical_models.config import ComputeConfig, Configuration
from diffwofost.physical_models.crop.wofost72 import Wofost72
from diffwofost.physical_models.engine import Engine
from diffwofost.physical_models.soil.classic_waterbalance import WaterbalancePP
from diffwofost.physical_models.crop.evapotranspiration import Evapotranspiration

warnings.filterwarnings("ignore", message="To copy construct from a tensor.*")
ComputeConfig.set_device("cpu")
ComputeConfig.set_dtype(torch.float64)

# Colab puts everything under /content. These paths line up with what the
# data-fetch cell below produces, and what the rest of the notebook expects.
data_dir      = Path("/content/data")
data_temp_dir = Path("/content/data_temp")
print(f"data_dir      = {data_dir}")
print(f"data_temp_dir = {data_temp_dir}")


### 2.1 Download data and pre-trained weights of the ML models

Three sources:

1. The field-trial files are mirrored verbatim from the
   [Harvard Dataverse dataset by Ten Den et al. (2024)](https://doi.org/10.7910/DVN/1LC6W7),
2. `models_bundle.zip` with the pre-trained `stress_nn_random.pt`
   (the hybrid) and `pure_lstm_random.pt` (the pure-ML reference).
3. PCSE stock files — config, crop YAML, and an agromanagement
   template, pulled from `ajwdewit/pcse` and `ajwdewit/pcse_notebooks`

In [ ]:
#@title Download data and pre-trained weights { display-mode: "form" }

for d in [
    data_temp_dir,
    data_temp_dir / "trained_models",
    data_dir / "conf",
    data_dir / "crop",
    data_dir / "agro",
]:
    d.mkdir(parents=True, exist_ok=True)

# 1. Field-trial data + pre-trained models — this tutorial repo's data/.
#    Field data: Ten Den et al. (2024), CC BY-NC-SA 4.0, mirrored from
#    Harvard Dataverse (doi:10.7910/DVN/1LC6W7). Models: derivative works,
#    same licence. See DATA_LICENSE.md.
TUTORIAL_DATA = "https://raw.githubusercontent.com/ronvree/diffwofost-tutorial/master/data"
for name in [
    "Plotspecific_processed.csv",
    "Weatherfile_lelystad.xlsx",
    "Weatherfile_vredepeel.xlsx",
]:
    dest = data_temp_dir / name
    if not dest.exists():
        print(f"Downloading {name}...")
        urlretrieve(f"{TUTORIAL_DATA}/{name}", dest)

models_dir = data_temp_dir / "trained_models"
zip_path = models_dir / "models_bundle.zip"
if not (models_dir / "stress_nn_random.pt").exists():
    print("Downloading pre-trained models...")
    urlretrieve(f"{TUTORIAL_DATA}/models_bundle.zip", zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(models_dir)
    zip_path.unlink()

# 2. PCSE stock files (Apache-2.0)
PCSE_URL = "https://raw.githubusercontent.com/ajwdewit/pcse/master/pcse/conf"
NB_URL   = "https://raw.githubusercontent.com/ajwdewit/pcse_notebooks/master/data"
for url, dest in [
    (f"{PCSE_URL}/Wofost72_PP.conf",     data_dir / "conf" / "Wofost72_PP.conf"),
    (f"{NB_URL}/crop/crops.yaml",        data_dir / "crop" / "crops.yaml"),
    (f"{NB_URL}/crop/potato.yaml",       data_dir / "crop" / "potato.yaml"),
    (f"{NB_URL}/agro/AGMT_C2_2020.agro", data_dir / "agro" / "AGMT_C2_2020.agro"),
]:
    if not dest.exists():
        print(f"Downloading {dest.name}...")
        urlretrieve(url, dest)

# 3. The 2019 agromanagement file is a one-line variant of the 2020 template
#    (different year). Generate it inline rather than maintaining a copy.
agro_2019 = data_dir / "agro" / "AGMT_C2_2019.agro"
if not agro_2019.exists():
    agro_2019.write_text(
        "Version: 1.0\n"
        "AgroManagement:\n"
        "- 2019-04-20:\n"
        "    CropCalendar:\n"
        "        crop_name: 'potato'\n"
        "        variety_name: 'Potato_C2_C5'\n"
        "        crop_start_date: 2019-04-20\n"
        "        crop_start_type: 'sowing'\n"
        "        crop_end_date: 2019-10-31\n"
        "        crop_end_type: 'maturity'\n"
        "        max_duration: 300\n"
        "    TimedEvents:\n"
        "    StateEvents:\n"
    )

# Convenient handles used by the rest of the notebook
conf_path = data_dir / "conf" / "Wofost72_PP.conf"
crop_path = data_dir / "crop"

print("\nAll data ready.")


In [ ]:
#@title Convert weather to PCSE format { display-mode: "form" }
# Convert both data_temp weather files into PCSE-compatible format
PCSE_COLS = ["DAY", "IRRAD", "TMIN", "TMAX", "VAP", "WIND", "RAIN", "SNOWDEPTH"]
UNITS = ["date", "kJ/m2/day or hours", "Celsius", "Celsius", "kPa", "m/sec", "mm", "cm"]


def convert_weather_to_pcse_format(src, dst, *, force=False):
    if dst.exists() and not force:
        return dst
    src_wb = openpyxl.load_workbook(src, data_only=True)
    sh = src_wb.active
    country, station, description = sh.cell(2, 2).value, sh.cell(3, 2).value, sh.cell(4, 2).value
    source_text, nodata_val = sh.cell(5, 2).value, sh.cell(6, 2).value
    longitude, latitude, elevation = sh.cell(8, 1).value, sh.cell(8, 2).value, sh.cell(8, 3).value
    angstrom_a, angstrom_b, has_sunshine = sh.cell(8, 4).value, sh.cell(8, 5).value, sh.cell(8, 6).value
    df = pd.read_excel(src, header=9).iloc[1:].reset_index(drop=True)[PCSE_COLS].copy()
    df["DAY"] = pd.to_datetime(df["DAY"])
    for c in PCSE_COLS[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df.loc[df["SNOWDEPTH"] < 0, "SNOWDEPTH"] = nodata_val

    wb = Workbook(); s = wb.active; s.title = "Weather"
    s.cell(1, 1).value = "Site Characteristics"
    s.cell(2, 1).value = "Country"; s.cell(2, 2).value = country
    s.cell(3, 1).value = "Station"; s.cell(3, 2).value = station
    s.cell(4, 1).value = "Description"; s.cell(4, 2).value = description
    s.cell(5, 1).value = "Source"; s.cell(5, 2).value = source_text
    s.cell(6, 1).value = "Contact"
    s.cell(7, 1).value = "Missing values"; s.cell(7, 2).value = nodata_val
    for j, l in enumerate(["Longitude", "Latitude", "Elevation", "AngstromA", "AngstromB", "HasSunshine"], 1):
        s.cell(8, j).value = l
    for j, v in enumerate([longitude, latitude, elevation, angstrom_a, angstrom_b, has_sunshine], 1):
        s.cell(9, j).value = v
    s.cell(10, 1).value = "Observed data"
    for j, c in enumerate(PCSE_COLS, 1):
        s.cell(11, j).value = c
    for j, u in enumerate(UNITS, 1):
        s.cell(12, j).value = u
    for i, row in enumerate(df.itertuples(index=False), 13):
        for j, v in enumerate(row, 1):
            s.cell(i, j).value = v.to_pydatetime() if hasattr(v, "to_pydatetime") else v
    wb.save(dst)
    return dst


weather_paths = {
    "L": convert_weather_to_pcse_format(
        data_temp_dir / "Weatherfile_lelystad.xlsx",
        data_temp_dir / "Weatherfile_lelystad_pcse.xlsx",
    ),
    "V": convert_weather_to_pcse_format(
        data_temp_dir / "Weatherfile_vredepeel.xlsx",
        data_temp_dir / "Weatherfile_vredepeel_pcse.xlsx",
    ),
}
print(f"Weather files ready: {list(weather_paths.values())}")


## 3. Data inspection

The dataset is a multi-year potato field trial run at two Dutch sites:

- **Lelystad (L)** — clay soil, central Netherlands;
- **Vredepeel (V)** — sandy soil, southeast Netherlands.

The experimental design is a factorial:

```
2 years (2019, 2020) × 2 sites × 6 cultivars × 3 N-levels × 2 W-levels
≈ 174 unique plot-years
```

Each plot has ~7 destructive biomass samples taken over the growing season,
plus continuous weather records. Let's load it up and have a look under the hood.


In [ ]:
#@title Inspect data { display-mode: "form" }
obs_df = pd.read_csv(data_temp_dir / "Plotspecific_processed.csv")
obs_df["Date"] = pd.to_datetime(obs_df["Date"])

# Drop the small Shadow side-experiment (C2 N2 W2 only on Lelystad, ~60 rows)
obs_df = obs_df[obs_df["Shadow"].isna()].copy()

# Keep only rows with at least one biomass/LAI observation
BIO_COLUMNS = ["LeavesDW", "StemDW", "tubersDW", "LAI"]
ROOT_COLUMN = "rootsDW"
obs_df = obs_df[obs_df[BIO_COLUMNS + [ROOT_COLUMN]].notna().any(axis=1)].copy()

# Per-plot index: (Year, Location, Plotnumber)
plot_keys = (
    obs_df[["Year", "Location", "Plotnumber", "Cultivar", "Nitrogen", "Irrigation"]]
    .drop_duplicates()
    .sort_values(["Year", "Location", "Plotnumber"])
    .reset_index(drop=True)
)
PLOT_KEYS = list(plot_keys.itertuples(index=False, name="Plot"))
print(f"Total plot-years (after Shadow drop): {len(PLOT_KEYS)}")
print()
print("Plot-years per (Year, Location, Cultivar):")
print(plot_keys.groupby(["Year", "Location", "Cultivar"]).size().unstack(fill_value=0))
print()
print("Per-variable observation counts:")
for col in BIO_COLUMNS + [ROOT_COLUMN]:
    print(f"  {col:<10} non-null: {obs_df[col].notna().sum()}")


## 4. Explore the data

Before fitting any model, a visual survey helps us sanity-check the dataset and
form intuition about what a model should be able to learn. Three quick views:

1. biomass / LAI trajectories per cultivar,
2. site-level comparison (Lelystad vs. Vredepeel),
3. treatment-effect preview (does N / W leave a signature?).


### 4.1 Biomass and LAI trajectories per cultivar

We expect leaves to peak around flowering, stems to plateau, tubers to rise
after flowering, and LAI to roughly follow the leaf curve. If anything looks
wildly off here, the data — not the model — is the first place to look.


In [ ]:
#@title Plot: biomass and LAI by cultivar { display-mode: "form" }
#@markdown §4.1 — mean ± std at each sample date, per year (no binning).

YEAR_STYLE = {2019: "-", 2020: "--"}

CULTIVAR_COLORS = {c: plt.cm.tab10(i) for i, c in
                   enumerate(sorted(obs_df["Cultivar"].dropna().unique()))}
PLOT_VARIABLES = [
    ("LeavesDW", "Leaf dry matter (g / m²)"),
    ("StemDW", "Stem dry matter (g / m²)"),
    ("tubersDW", "Tuber dry matter (g / m²)"),
    ("LAI", "Leaf area index"),
]


def mean_std_by_year(df, group_col, value_col):
    """Mean ± std across plots at each exact sample date (no binning); per year."""
    sub = df[[group_col, value_col, "Year", "Date"]].dropna(subset=[value_col]).copy()
    sub["Date"] = pd.to_datetime(sub["Date"]).dt.normalize()
    agg = (
        sub.groupby(["Year", group_col, "Date"], observed=True)[value_col]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
    )
    agg.loc[agg["n"] < 2, "std"] = 0.0
    agg["std"] = agg["std"].fillna(0.0)
    return agg


def plot_mean_std(ax, agg, group_col, colors, value_label=None):
    for (year, group), g in agg.groupby(["Year", group_col]):
        g = g.sort_values("Date")
        color = colors.get(group, "gray")
        ls = YEAR_STYLE.get(int(year), "-")
        ax.plot(g["Date"], g["mean"], color=color, ls=ls, lw=1.8)
        ax.fill_between(
            g["Date"], g["mean"] - g["std"], g["mean"] + g["std"],
            color=color, alpha=0.15, linewidth=0,
        )
    if value_label:
        ax.set_ylabel(value_label)
    ax.tick_params(axis="x", rotation=30)


fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=False)
for ax, (col, title) in zip(axes.ravel(), PLOT_VARIABLES):
    agg = mean_std_by_year(obs_df, "Cultivar", col)
    plot_mean_std(ax, agg, "Cultivar", CULTIVAR_COLORS)
    ax.set_title(title)
    ax.grid(alpha=0.3)

from matplotlib.lines import Line2D
cultivar_handles = [
    Line2D([0], [0], color=CULTIVAR_COLORS[c], lw=2, label=c)
    for c in sorted(CULTIVAR_COLORS)
]
year_handles = [
    Line2D([0], [0], color="k", ls=ls, lw=2, label=str(year))
    for year, ls in YEAR_STYLE.items()
]
axes[0, 0].legend(handles=cultivar_handles + year_handles, loc="upper left", fontsize=7, ncol=2)
plt.tight_layout(); plt.show()


### 4.2 Site comparison

Lelystad vs. Vredepeel side by side. If the two sites differ enough that they
look like distinct populations, our model needs to be told which is which (the
site bit in the input features handles this).


In [ ]:
#@title Plot: site comparison { display-mode: "form" }
#@markdown §4.2 — Lelystad vs Vredepeel; mean ± std at each sample date, per year.

from matplotlib.lines import Line2D

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=False)
loc_colors = {"L": "tab:blue", "V": "tab:orange"}
loc_labels = {"L": "Lelystad", "V": "Vredepeel"}
for ax, (col, title) in zip(axes.ravel(), PLOT_VARIABLES):
    agg = mean_std_by_year(obs_df, "Location", col)
    plot_mean_std(ax, agg, "Location", loc_colors)
    ax.set_title(title)
    ax.grid(alpha=0.3)

loc_handles = [
    Line2D([0], [0], color=loc_colors[k], lw=2, label=loc_labels[k])
    for k in ["L", "V"]
]
year_handles = [
    Line2D([0], [0], color="k", ls=ls, lw=2, label=str(year))
    for year, ls in YEAR_STYLE.items()
]
axes[0, 0].legend(handles=loc_handles + year_handles, loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


### 4.3 Treatment-effect preview

Do nitrogen level and irrigation treatment leave a visible mark on the
observations? If they do, the hybrid model has something to learn from them —
the NN gets a one-hot encoding of treatment as input.

> ❓ **Discuss with a neighbour**
>
> Look at the two panels below. Can you see the treatment effect?
>


In [ ]:
#@title Plot: treatment-effect preview { display-mode: "form" }
#@markdown §4.3 — nitrogen and irrigation; mean ± std at each sample date, per year.

from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False)
n_colors = {n: plt.cm.viridis(i / 2) for i, n in enumerate(sorted(obs_df["Nitrogen"].dropna().unique()))}
w_colors = {w: plt.cm.plasma(i) for i, w in enumerate(sorted(obs_df["Irrigation"].dropna().unique()))}

agg_n = mean_std_by_year(obs_df, "Nitrogen", "tubersDW")
plot_mean_std(axes[0], agg_n, "Nitrogen", n_colors)
axes[0].set_title("Tuber dry matter by N level")
axes[0].set_ylabel("tubersDW (g / m²)")
axes[0].grid(alpha=0.3)

agg_w = mean_std_by_year(obs_df, "Irrigation", "LAI")
plot_mean_std(axes[1], agg_w, "Irrigation", w_colors)
axes[1].set_title("LAI by irrigation level")
axes[1].set_ylabel("LAI")
axes[1].grid(alpha=0.3)

n_handles = [Line2D([0], [0], color=n_colors[n], lw=2, label=str(n)) for n in sorted(n_colors)]
w_handles = [Line2D([0], [0], color=w_colors[w], lw=2, label=str(w)) for w in sorted(w_colors)]
year_handles = [
    Line2D([0], [0], color="k", ls=ls, lw=2, label=str(year))
    for year, ls in YEAR_STYLE.items()
]
axes[0].legend(handles=n_handles + year_handles, loc="upper left", fontsize=8)
axes[1].legend(handles=w_handles + year_handles, loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


## 5. The hybrid model

The hybrid we will build looks like this (simplified):

```
                +-------------------+      +-------------------+
   weather ---> | Phenology (DVS)   | ---> | Partitioning      |
   crop YAML    | (WOFOST physics)  |      | (WOFOST physics)  |
                +-------------------+      +-------------------+
                                                    |
                                                    v
                  +------------------+      +---------------------+
   weather ---->  | Stress NN        | -->  | Gross assimilation  |
   DVS, treatment | (learned RFTRA)  |      | (WOFOST × RFTRA)    |
                  +------------------+      +---------------------+
                                                    |
                                                    v
                                            organ biomass per day
                                                    |
                                            compare to observations
                                                    |
                                            loss --> backprop --> NN
```
**How it works.** In standard WOFOST, the daily stress reduction factor `RFTRA` is computed inside the evapotranspiration module. This factor represents how much crop assimilation should be reduced due to limited transpiration under water stress.
Under the `_PP` (potential production) configuration, however, no water stress
is simulated, and the model simply returns `RFTRA = 1.0` (no stress).

In this tutorial, we replace that evapotranspiration stress component with a
small neural-network module called `NNStressFactor`.

Each simulated day, the WOFOST engine calls this module to ask:
> “By how much should today’s gross assimilation be reduced?”


The replacement module:

1. assembles the day’s feature vector
(weather variables, DVS, and plot/treatment context),
2. feeds those inputs through a small multilayer perceptron (StressNN),
3. returns a sigmoid output as `RFTRA ∈ [0, 1]`.


Importantly, we use this learned stress factor not only for water stress, but
also implicitly for nutrient limitations such as nitrogen stress. This is a
modelling shortcut: physiologically, RFTRA was originally intended only for
transpiration-related stress. However, because it acts as a multiplicative
reduction on gross assimilation, it can also serve as an effective proxy for
other stressors that reduce canopy carbon fixation.

Because the entire simulation engine is implemented in PyTorch, every operation
remains differentiable. The seasonal loss (for example, yield prediction error)
therefore stays connected to the neural-network parameters through the complete
computational graph of the crop model.

Calling `loss.backward()` propagates gradients backward through the entire growing season and into the weights of `StressNN` — which is the central idea behind diffWOFOST.


### 5.1 The two plug-in components

`StressNN` is the tiny MLP that maps daily features → `RFTRA ∈ [0, 1]`.
`NNStressFactor` is the WOFOST evapotranspiration swap that calls it on every
simulated day.

The implementations are included below so you can see exactly how the hybrid hook works. Skim the docstrings if you like — or skip straight to §6, which explains what goes into the network.


In [ ]:
import torch

from diffwofost.physical_models.config import ComputeConfig


class StressNN(torch.nn.Module):
    """Tiny MLP mapping per-day features to an `RFTRA`-style stress factor.

    Architecture: `n_features → hidden_size → 1` with a `SiLU` activation in
    between and a `sigmoid` output. The output layer is biased at
    initialization toward `sigmoid(2.2) ≈ 0.90`, so an untrained network
    starts at a *mild stress* level (about 90% of PP). This avoids two
    pathologies: (a) collapsing to zero biomass before the optimizer can
    learn anything, and (b) starting in the saturated tail of the sigmoid
    where the gradient `sigmoid'(5) ≈ 0.007` would dampen all updates by
    two orders of magnitude. At bias=2.2 the gradient is `sigmoid'(2.2) ≈
    0.087` — roughly 13× larger — so the optimizer can actually move.

    Args:
        n_features (int): Length of the input feature vector.
        hidden_size (int): Width of the hidden layer. Defaults to 16.
        init_no_stress (bool): If True, biases the output toward ~0.90 at
            initialization (mild stress, with usable gradient). Defaults to
            True.
    """

    def __init__(self, n_features, hidden_size=16, init_no_stress=True):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(n_features, hidden_size),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_size, 1),
        )
        if init_no_stress:
            with torch.no_grad():
                # bias=2.2 -> sigmoid(2.2) ~ 0.90, well outside the saturated
                # tail of the sigmoid so backprop gradients aren't dampened.
                # Output-layer weights are left at xavier defaults so the
                # network has full dynamic range to deviate from the bias.
                self.layers[-1].bias.fill_(2.2)
        self.to(device=ComputeConfig.get_device(), dtype=ComputeConfig.get_dtype())

    def forward(self, features):
        """Compute the stress factor from a feature vector.

        Args:
            features (torch.Tensor): 1-D tensor of input features.

        Returns:
            torch.Tensor: Scalar tensor in `[0, 1]` representing the stress
            factor (`RFTRA`).
        """
        out = self.layers(features)
        return torch.sigmoid(out).squeeze(-1)


In [ ]:
import torch
from pcse.traitlets import Instance

from diffwofost.physical_models.crop.evapotranspiration import Evapotranspiration


class NNStressFactor(Evapotranspiration):
    """Standard non-layered evapotranspiration with NN-overridden `RFTRA`.

    Runs the parent's `calc_rates` to populate `TRAMX`, `EVWMX`, `EVSMX`,
    `RFWS`, `RFOS`, then overrides `RFTRA` with `nn_model(features)` and
    recomputes `TRA = TRAMX × RFTRA`. The crop module reads `k.RFTRA` from
    the kiosk and applies it as `GASS = PGASS × k.RFTRA`, so the stress
    factor enters assimilation transparently.

    The feature builder is supplied as a callable rather than baked in,
    so the same engine class can be used with different feature sets.

    **Inputs from kiosk** (read indirectly by the parent's calc_rates and
    by the supplied `feature_builder`):

    | Name | Description                              |
    |------|------------------------------------------|
    | DVS  | Crop development stage                   |
    | LAI  | Leaf area index                          |
    | SM   | Soil moisture content                    |

    **Outputs to kiosk** (overridden):

    | Name  | Description                              |
    |-------|------------------------------------------|
    | RFTRA | NN-learned stress factor on assimilation |
    | TRA   | Crop transpiration rate (TRAMX × RFTRA)  |
    """

    nn_model = Instance(torch.nn.Module)
    feature_builder = Instance(object)

    def initialize(self, day, kiosk, parvalues, nn_model, feature_builder, shape=None):
        """Initialize using the parent's ET logic and attach the NN.

        Args:
            day: Start date of the simulation.
            kiosk (VariableKiosk): Variable kiosk of this PCSE instance.
            parvalues (ParameterProvider): Parameter provider for the ET module.
            nn_model (torch.nn.Module): Network mapping per-day feature
                vector to `RFTRA` in `[0, 1]`. Must return a scalar tensor.
            feature_builder (callable): Called daily as
                `feature_builder(day, drv, kiosk)` to produce a 1-D feature
                tensor for `nn_model`.
            shape (tuple | None): Target shape for state and rate variables.
        """
        super().initialize(day, kiosk, parvalues, shape=shape)
        self.nn_model = nn_model
        self.feature_builder = feature_builder

    def calc_rates(self, day=None, drv=None):
        """Compute ET rates with `RFTRA` replaced by the NN output."""
        super().calc_rates(day, drv)
        features = self.feature_builder(day, drv, self.kiosk)
        rftra = self.nn_model(features)
        if rftra.dim() > 0 and rftra.numel() == 1:
            rftra = rftra.reshape(())
        self.rates.RFTRA = rftra
        self.rates.TRA = self.rates.TRAMX * rftra


## 6. Building input features for the NN

The NN gets three groups of inputs:

**Per-plot context (constant over the season).**

| Field | Encoding | Dim |
|-------|----------|-----|
| Site | binary (Lelystad=0, Vredepeel=1) | 1 |
| Nitrogen level | numeric (N0=0.0, N1=0.3, N2=1.3) | 1 |
| Irrigation level | binary (W1=0, W2=1) | 1 |
| Cultivar | one-hot of [C1, C2, C3, C4, C5, C6] | 6 |

**Per-day weather features (computed once per site).**

| Field | Computed from | Dim |
|-------|---------------|-----|
| VPD (vapor pressure deficit) | `SVP(TMAX) − VAP` | 1 |
| TMAX (max temp) | weather file | 1 |
| 7-day rolling rainfall | weather file | 1 |
| IRRAD (radiation) | weather file (÷ 1e4) | 1 |

**Per-day crop-state features (read from the engine kiosk daily).**

| Field | Dim |
|-------|-----|
| DVS | 1 |

**Total NN input dim: 4 (weather) + 1 (DVS) + 9 (plot context) = 14.**



In [ ]:
#@title Feature encodings: plot context { display-mode: "form" }
#@markdown §6 — site, N level, irrigation, and cultivar one-hot encodings.

SITE_INDEX = {"L": 0, "V": 1}
N_LEVEL_NUMERIC = {"N0": 0.0, "N1": 0.3, "N2": 1.3}
W_LEVEL_INDEX = {"W1": 0, "W2": 1}
CULTIVARS = ["C1", "C2", "C3", "C4", "C5", "C6"]
CULTIVAR_INDEX = {c: i for i, c in enumerate(CULTIVARS)}


def make_plot_context_tensor(cultivar, nitrogen, irrigation, location):
    site_bit = float(SITE_INDEX[location])
    n_num = float(N_LEVEL_NUMERIC[nitrogen])
    w_bit = float(W_LEVEL_INDEX[irrigation])
    cultivar_oh = [0.0] * len(CULTIVARS)
    cultivar_oh[CULTIVAR_INDEX[cultivar]] = 1.0
    return torch.tensor(
        [site_bit, n_num, w_bit] + cultivar_oh,
        dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
    )


PLOT_CONTEXT_DIM = 3 + len(CULTIVARS)
print(f"Plot context dim: {PLOT_CONTEXT_DIM}")
print(f"Example (C2, N2, W2, Lelystad): {make_plot_context_tensor('C2', 'N2', 'W2', 'L')}")


In [ ]:
#@title Feature encodings: weather { display-mode: "form" }
#@markdown §6 — VPD, rolling rainfall, and other per-day weather features.

def saturation_vp(temp_c):
    return 0.6108 * np.exp(17.27 * temp_c / (temp_c + 237.3))


def precompute_weather_features(weather_path):
    df = pd.read_excel(weather_path, header=10).iloc[1:].reset_index(drop=True)
    df["DAY"] = pd.to_datetime(df["DAY"])
    for c in ["IRRAD", "TMIN", "TMAX", "VAP", "WIND", "RAIN", "SNOWDEPTH"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.sort_values("DAY").reset_index(drop=True)
    for c in ["RAIN", "TMAX", "VAP", "IRRAD"]:
        df.loc[df[c] <= -990, c] = np.nan
    df["RAIN"] = df["RAIN"].fillna(0.0)
    df["VPD"] = (saturation_vp(df["TMAX"]) - df["VAP"]).clip(lower=0.0)
    df["RAIN_ROLL_7D"] = df["RAIN"].rolling(7, min_periods=1).sum()
    df["IRRAD_NORM"] = df["IRRAD"] / 1e4

    features = {}
    for _, row in df.iterrows():
        t = pd.Timestamp(row["DAY"]).normalize()
        features[t] = torch.tensor(
            [row["VPD"], row["TMAX"], row["RAIN_ROLL_7D"], row["IRRAD_NORM"]],
            dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
        )
    return features


WEATHER_FEATURES = {
    "L": precompute_weather_features(weather_paths["L"]),
    "V": precompute_weather_features(weather_paths["V"]),
}
WEATHER_FEATURE_DIM = 4
print(f"Weather feature dim: {WEATHER_FEATURE_DIM}")
print(f"Lelystad features pre-computed for {len(WEATHER_FEATURES['L'])} days")
print(f"Vredepeel features pre-computed for {len(WEATHER_FEATURES['V'])} days")


Each simulated day, WOFOST asks a small helper for the neural network’s inputs. It passes the date, that day’s weather, and the current crop state (including DVS, read from the engine’s shared variable store). The helper returns one combined feature vector.


In [ ]:
#@title Daily feature assembler { display-mode: "form" }
#@markdown §6 — combines weather, DVS, and plot context for the stress NN.

class PlotFeatureBuilder:
    """Daily feature assembler for a specific plot.

    Pre-computed weather features and the plot context tensor are baked in at
    construction; only DVS is read fresh from the kiosk on each call.
    """

    def __init__(self, weather_features_by_date, plot_context):
        self.weather_features = weather_features_by_date
        self.plot_context = plot_context

    def __call__(self, day, drv, kiosk):
        wf = self.weather_features.get(pd.Timestamp(day).normalize())
        if wf is None:
            wf = torch.zeros(WEATHER_FEATURE_DIM, dtype=ComputeConfig.get_dtype(),
                             device=ComputeConfig.get_device())
        dvs = kiosk["DVS"] if "DVS" in kiosk else torch.tensor(0.0)
        if not isinstance(dvs, torch.Tensor):
            dvs = torch.tensor(dvs, dtype=ComputeConfig.get_dtype())
        crop_state = dvs.flatten()[:1]
        return torch.cat([wf, crop_state, self.plot_context])


N_FEATURES = WEATHER_FEATURE_DIM + 1 + PLOT_CONTEXT_DIM
print(f"Total NN input dim: {N_FEATURES}")


> ❓ **Discussion question**
>
> Crop stress is often cumulative: a week of low rain matters, not just today’s VPD. The hybrid stress NN is fed mostly same-day inputs (weather, DVS, treatment), except for one explicit history term — 7-day rolling rainfall.
>
> Does that mix feel sufficient, or too “memoryless” for drought and N stress?
> If stress should depend on what happened earlier in the season, what would you add? (Rolling windows? soil moisture from WOFOST? A recurrent network like the LSTM in §13?)
>


## 7. Engine setup

We use the standard `Wofost72_PP` stack — phenology, partitioning,
respiration, leaf dynamics, the works — but with a single targeted swap:
`evapotranspiration` → `NNStressFactor`. Everything else runs unchanged.


In [ ]:
#@title Engine setup
#@markdown §7 — WOFOST72_PP stack with `NNStressFactor` plugged into evapotranspiration.

crop_data_provider = YAMLCropDataProvider(fpath=crop_path, force_reload=True)
parameter_provider = ParameterProvider(
    cropdata=crop_data_provider,
    soildata=DummySoilDataProvider(),
    sitedata=WOFOST72SiteDataProvider(WAV=10),
)

OUTPUT_VARS = ["DVS", "LAI", "TAGP", "WLV", "TWST", "TWSO", "TWRT", "RFTRA"]
base_pcse_config = Configuration.from_pcse_config_file(conf_path)


def build_config(et_class=None, et_kwargs=None):
    cfg = Configuration(
        CROP=Wofost72, SOIL=WaterbalancePP,
        AGROMANAGEMENT=base_pcse_config.AGROMANAGEMENT,
        OUTPUT_VARS=OUTPUT_VARS.copy(),
        SUMMARY_OUTPUT_VARS=list(base_pcse_config.SUMMARY_OUTPUT_VARS),
        TERMINAL_OUTPUT_VARS=list(base_pcse_config.TERMINAL_OUTPUT_VARS),
        OUTPUT_INTERVAL=base_pcse_config.OUTPUT_INTERVAL,
        OUTPUT_INTERVAL_DAYS=base_pcse_config.OUTPUT_INTERVAL_DAYS,
        OUTPUT_WEEKDAY=base_pcse_config.OUTPUT_WEEKDAY,
        model_config_file=base_pcse_config.model_config_file,
        description=base_pcse_config.description,
    )
    if et_class is not None:
        comp = {"class": et_class}
        if et_kwargs:
            comp.update(et_kwargs)
        cfg.CROP_COMPONENTS["evapotranspiration"] = comp
    return cfg


reference_config = build_config()  # default WOFOST72_PP, no NN

agro_paths = {
    2019: data_dir / "agro" / "AGMT_C2_2019.agro",
    2020: data_dir / "agro" / "AGMT_C2_2020.agro",
}


def results_to_tensors(results, var_names=OUTPUT_VARS):
    out = {"day": [pd.Timestamp(r["day"]) for r in results]}
    for v in var_names:
        out[v] = torch.stack([
            torch.as_tensor(r[v], dtype=ComputeConfig.get_dtype(),
                            device=ComputeConfig.get_device())
            for r in results
        ])
    return out


# Cache weather data providers (one per site)
WEATHER_DATA_PROVIDERS = {
    site: ExcelWeatherDataProvider(str(weather_paths[site]))
    for site in ["L", "V"]
}


def run_engine(config, year, location, weather):
    eng = Engine(config=config)
    eng.setup(copy.deepcopy(parameter_provider), weather, YAMLAgroManagementReader(agro_paths[year]))
    eng.run_till_terminate()
    return results_to_tensors(eng.get_output())


def run_plot_with_nn(plot, nn_model):
    feature_builder = PlotFeatureBuilder(
        WEATHER_FEATURES[plot.Location],
        make_plot_context_tensor(plot.Cultivar, plot.Nitrogen, plot.Irrigation, plot.Location),
    )
    cfg = build_config(NNStressFactor, et_kwargs={
        "nn_model": nn_model, "feature_builder": feature_builder,
    })
    return run_engine(cfg, plot.Year, plot.Location, WEATHER_DATA_PROVIDERS[plot.Location])


def run_plot_reference(plot):
    return run_engine(reference_config, plot.Year, plot.Location,
                      WEATHER_DATA_PROVIDERS[plot.Location])


print("Engine helpers ready.")


## 8. Prepare for training

Three preparatory pieces: a loss function, a reference-WOFOST baseline run (to
normalise the loss), and a train/test split.


### 8.1 Loss function

A pooled normalised RMSE over the four observed variables (`WLV`, `TWST`,
`TWSO`, `LAI`) and `TWRT`. "Normalised" means dividing by the mean of the
observations, so each variable is on a comparable scale before weights are
applied.


In [ ]:
#@title Loss function helpers { display-mode: "form" }
#@markdown Pooled normalised RMSE over biomass/LAI observations.

OBS_TO_PCSE = {"LeavesDW": "WLV", "StemDW": "TWST", "tubersDW": "TWSO", "LAI": "LAI"}
ROOT_PCSE = "TWRT"
VAR_BASE_WEIGHTS = {"WLV": 1.0, "TWST": 1.0, "TWSO": 1.0, "LAI": 1.0, "TWRT": 0.5}


def get_plot_observations(plot, simulation_days):
    rows = obs_df[
        (obs_df["Year"] == plot.Year)
        & (obs_df["Location"] == plot.Location)
        & (obs_df["Plotnumber"] == plot.Plotnumber)
    ].copy()
    keep = rows[BIO_COLUMNS + [ROOT_COLUMN]].notna().any(axis=1)
    rows = rows.loc[keep].sort_values("Date").reset_index(drop=True)

    lookup = {pd.Timestamp(d).normalize(): i for i, d in enumerate(simulation_days)}
    last_idx = len(simulation_days) - 1
    last_day = pd.Timestamp(simulation_days[-1]).normalize()
    first_day = pd.Timestamp(simulation_days[0]).normalize()
    matched = []
    for _, r in rows.iterrows():
        key = pd.Timestamp(r["Date"]).normalize()
        if key in lookup:
            matched.append((lookup[key], r))
        elif key > last_day:
            matched.append((last_idx, r))
        elif key < first_day:
            matched.append((0, r))
    if not matched:
        return None, None
    indices = torch.tensor([m[0] for m in matched], dtype=torch.long)
    df_m = pd.DataFrame([m[1] for m in matched])
    targets = {p: torch.tensor(df_m[obs_col].to_numpy(),
                               dtype=ComputeConfig.get_dtype(),
                               device=ComputeConfig.get_device())
               for obs_col, p in OBS_TO_PCSE.items()}
    if df_m[ROOT_COLUMN].notna().any():
        targets[ROOT_PCSE] = torch.tensor(df_m[ROOT_COLUMN].to_numpy(),
                                          dtype=ComputeConfig.get_dtype(),
                                          device=ComputeConfig.get_device())
    return indices, targets


def per_plot_loss(results, targets, indices, weights):
    tl = torch.zeros((), dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    diag = {}
    for name, target in targets.items():
        pred = results[name].index_select(0, indices)
        valid = torch.isfinite(target)
        if not torch.any(valid):
            continue
        pred = pred[valid]; target_valid = target[valid]
        scale = torch.mean(target_valid).abs().clamp_min(1e-6)
        rmse = torch.sqrt(torch.mean(((pred - target_valid) / scale) ** 2))
        tl = tl + weights[name] * rmse
        diag[name] = rmse.detach().cpu().item()
    return tl, diag


def pooled_loss(plots, runner, weights):
    total = torch.zeros((), dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    n_used = 0
    all_diag = {k: [] for k in weights}
    per_plot = {}
    for p in plots:
        try:
            r = runner(p)
        except Exception as exc:
            print(f"  WARN: failed on plot {p}: {exc}")
            continue
        idx, tgt = get_plot_observations(p, r["day"])
        if idx is None:
            continue
        pl, diag = per_plot_loss(r, tgt, idx, weights)
        total = total + pl
        n_used += 1
        per_plot[(p.Year, p.Location, p.Plotnumber)] = r
        for k, v in diag.items():
            all_diag[k].append(v)
    avg_diag = {k: float(np.mean(v)) if v else float("nan") for k, v in all_diag.items()}
    return total / max(n_used, 1), avg_diag, per_plot, n_used


print("Loss machinery ready.")


### 8.2 Reference baseline + normalised weights

We first run the *default* WOFOST72_PP on every plot — no NN, no stress — to
get a baseline error per variable. 

> **Important caveat about this baseline.** The reference WOFOST72_PP run here
> is *uncalibrated* for this dataset. It uses the stock potato parameters from
> the PCSE distribution and a generic agro-management file derived from one
> cultivar (Fontane / C2). A properly calibrated WOFOST run would fit per-cultivar
> genetic parameters (`SPAN`, `TSUM1`, `TSUM2`, etc.) and the correct sowing
> dates per plot. 
> For our purposes here, the uncalibrated run serves two functions: 
> (a) a *visual reference line* on plots so we can see what the NN is correcting.
> (b) a sensible loss normaliser: The loss combines several variables on very different scales
> (e.g. tuber DM vs LAI). We rescale each term so no single variable dominates training. 
> Note: We are **not** claiming the hybrid is "better than WOFOST" in any
> apples-to-apples sense.

In [ ]:
print(f"Running reference WOFOST72_PP on {len(PLOT_KEYS)} plot-years...")
ref_loss_uniform, ref_diag, REFERENCE_PLOT_RESULTS, n_ref_used = pooled_loss(
    PLOT_KEYS, run_plot_reference, VAR_BASE_WEIGHTS,
)
print(f"  Reference plots successfully simulated: {n_ref_used}/{len(PLOT_KEYS)}")
print(f"  Reference pooled loss (uniform weights): {ref_loss_uniform.item():.4f}")
print()
print("Per-variable RMSE (averaged across plots):")
for k, v in ref_diag.items():
    print(f"  {k:<6} {v:.4f}")

NORMALIZED_WEIGHTS = {
    k: VAR_BASE_WEIGHTS[k] / max(ref_diag[k], 0.1) for k in VAR_BASE_WEIGHTS
}
print()
print("Normalised loss weights (used during training):")
for k, v in NORMALIZED_WEIGHTS.items():
    print(f"  {k:<6} {v:.3f}")


### 8.3 Train/test split

We use a simple random 80/20 split of the plot-years. This tests
*interpolation within the design space*: training and test see the same
cultivars, sites, and treatments — just different plots.

In [ ]:
#@title Train/test split { display-mode: "form" }

def split_random(plots, test_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(plots))
    n_test = int(round(len(plots) * test_fraction))
    test_idx = set(idx[:n_test].tolist())
    train, test = [], []
    for i, p in enumerate(plots):
        (test if i in test_idx else train).append(p)
    return train, test


train_plots, test_plots = split_random(PLOT_KEYS)
print(f"  train plot-years: {len(train_plots)}")
print(f"  test  plot-years: {len(test_plots)}")


## 9. Initialise the stress neural network

`StressNN` is a tiny MLP — 14 → 16 → 1 with a `SiLU` non-linearity and a
sigmoid output. With `init_no_stress=True`, the output bias is set so the
*untrained* network returns `RFTRA ≈ 0.90` (mild stress). This avoids two
pathologies:

- starting at `RFTRA = 0` would kill all biomass before the optimizer could
  learn anything;
- starting in the saturated tail of the sigmoid (e.g. `RFTRA = 0.99`) would
  shrink the gradient to almost nothing.

13× larger gradient at init = an optimizer that can actually move.


In [ ]:
torch.manual_seed(7)
stress_nn = StressNN(n_features=N_FEATURES, hidden_size=16, init_no_stress=True)
n_params = sum(p.numel() for p in stress_nn.parameters())
print(f"StressNN: {n_params} parameters")
print(stress_nn)


## 10. Train the model

Each training step does the following for every plot in the train set:

1. **Forward**: run the engine for the whole season. Every simulated day, the
   `NNStressFactor` component evaluates the NN to produce `RFTRA`, which
   reduces the day's gross assimilation.
2. **Compute loss**: extract simulated biomass at observation dates, compare to
   measurements, accumulate the normalised RMSE.
3. **Backward**: call `loss.backward()`. PyTorch propagates gradients through
   every operation in the engine — leaf growth, partitioning, root
   redistribution — and ends up at the NN parameters.
4. **Optimise**: Adam takes a step.

That's it. The engine looks like a normal PyTorch module from the optimizer's
point of view.

**Training is expensive** (each step = N engine runs × ~150 days each). To
keep this tutorial snappy we load a pre-trained checkpoint by default. Flip
`FORCE_RETRAIN = True` if you'd like to retrain from scratch (about 5–15 min
on CPU).


In [ ]:
FORCE_RETRAIN = False
MODEL_DIR = data_temp_dir / "trained_models"
MODEL_DIR.mkdir(exist_ok=True)
model_path = MODEL_DIR / "stress_nn_random.pt"

training_config = {
    "lr": 0.02,
    "max_steps": 60,
    "patience": 15,
    "min_delta": 5e-4,
}


def train_stress_nn(stress_nn, train_plots, test_plots, weights, cfg):
    optimizer = torch.optim.Adam(stress_nn.parameters(), lr=cfg["lr"])

    def runner(plot):
        return run_plot_with_nn(plot, stress_nn)

    train_history, test_history, diag_history = [], [], []
    best_loss, best_step = float("inf"), -1
    best_state = copy.deepcopy(stress_nn.state_dict())

    for step in range(cfg["max_steps"]):
        optimizer.zero_grad()
        train_loss, train_diag, _, _ = pooled_loss(train_plots, runner, weights)
        train_history.append(train_loss.detach().cpu().item())
        diag_history.append(train_diag)

        with torch.no_grad():
            test_loss, _, _, _ = pooled_loss(test_plots, runner, weights)
        test_history.append(test_loss.detach().cpu().item())

        if step % 2 == 0:
            print(f"  step {step:03d} | train={train_history[-1]:.4f} test={test_history[-1]:.4f}")

        if train_history[-1] < best_loss - cfg["min_delta"]:
            best_loss = train_history[-1]; best_step = step
            best_state = copy.deepcopy(stress_nn.state_dict())

        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(stress_nn.parameters(), max_norm=1.0)
        optimizer.step()

        if step - best_step >= cfg["patience"]:
            print(f"  early stopping at step {step}")
            break

    stress_nn.load_state_dict(best_state)
    return {
        "train_history": train_history,
        "test_history": test_history,
        "diag_history": diag_history,
    }


if model_path.exists() and not FORCE_RETRAIN:
    print(f"Loading saved model from {model_path.name}")
    saved = torch.load(model_path, weights_only=False)
    stress_nn.load_state_dict(saved["state_dict"])
    training_run = saved["training_run"]
    print(f"  Previously trained for {len(training_run['train_history'])} steps")
    print(f"  Saved train loss: {training_run['train_history'][-1]:.4f}")
    print(f"  Saved test  loss: {training_run['test_history'][-1]:.4f}")
    print(f"  (Set FORCE_RETRAIN = True above to retrain.)")
else:
    print("Training from scratch — this will take a while...")
    training_run = train_stress_nn(stress_nn, train_plots, test_plots, NORMALIZED_WEIGHTS, training_config)
    torch.save({
        "state_dict": stress_nn.state_dict(),
        "training_run": training_run,
        "n_features": N_FEATURES,
    }, model_path)
    print(f"Saved trained model to {model_path}")

# Final evaluation — collect simulation outputs for every plot (train + test)
with torch.no_grad():
    final_train_loss, final_train_diag, train_results, _ = pooled_loss(
        train_plots, lambda p: run_plot_with_nn(p, stress_nn), NORMALIZED_WEIGHTS,
    )
    final_test_loss, final_test_diag, test_results, _ = pooled_loss(
        test_plots, lambda p: run_plot_with_nn(p, stress_nn), NORMALIZED_WEIGHTS,
    )

print()
print(f"FINAL: train loss={final_train_loss.item():.4f}  test loss={final_test_loss.item():.4f}")


## 11. Inspect model performance

We inspect five things:

1. **Loss curves** — did training converge? Does test track train?
2. **Per-plot fits** — does the model recover the shape of the trajectories?
3. **Learned stress factor per plot** — what does `RFTRA(t)` look like for a few plots?
4. **Stress profile by cultivar** — does the NN learn cultivar-specific stress?
5. **Stress profile by nitrogen treatment** — does it learn the N effect?


### 11.1 Loss curves


In [ ]:
#@title Plot: training loss curves { display-mode: "form" }
#@markdown §11.1 — train vs test loss over steps.

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(training_run["train_history"], label="train", linewidth=2)
ax.plot(training_run["test_history"], label="test", linewidth=2, linestyle="--")
ax.set_title("Pooled normalised loss (random 80/20 split)")
ax.set_xlabel("training step")
ax.set_ylabel("loss")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


### 11.2 Per-plot fits

For one plot per cultivar in the test set, we overlay:

- the default WOFOST72_PP simulation (solid),
- the trained hybrid simulation (dashed),
- the actual field observations (black dots).


In [ ]:
#@title Plot: per-plot model fits { display-mode: "form" }
#@markdown §11.2 — hybrid, WOFOST reference, and observations.

def _select_representative_plots(plots, n_per_cultivar=1):
    by_c = {}
    for p in plots:
        by_c.setdefault(p.Cultivar, []).append(p)
    selected = []
    for c in CULTIVARS:
        if c in by_c:
            selected.extend(by_c[c][:n_per_cultivar])
    return selected


display_plots = _select_representative_plots(test_plots, n_per_cultivar=1)
if not display_plots:
    display_plots = _select_representative_plots(train_plots, n_per_cultivar=1)
print(f"Showing fits for {len(display_plots)} plots (one per cultivar in the test set)")


def _plot_observed(ax, plot, obs_col):
    sub = obs_df[
        (obs_df["Year"] == plot.Year)
        & (obs_df["Location"] == plot.Location)
        & (obs_df["Plotnumber"] == plot.Plotnumber)
    ]
    ax.scatter(sub["Date"], sub[obs_col], s=28, color="black", zorder=5, label="Observed")


fig, axes = plt.subplots(len(display_plots), 4, figsize=(20, 3 * len(display_plots)),
                          squeeze=False)
PLOT_VARS = [("WLV", "LeavesDW", "Leaf DM"),
             ("TWST", "StemDW", "Stem DM"),
             ("TWSO", "tubersDW", "Tuber DM"),
             ("LAI", "LAI", "LAI")]
for row, plot in enumerate(display_plots):
    key = (plot.Year, plot.Location, plot.Plotnumber)
    fitted = test_results.get(key) or train_results.get(key)
    ref = REFERENCE_PLOT_RESULTS.get(key)
    label = f"{plot.Cultivar}@{plot.Location}, {plot.Nitrogen}, {plot.Irrigation}, {plot.Year}"
    for col, (var, obs_col, title) in enumerate(PLOT_VARS):
        ax = axes[row, col]
        if ref is not None:
            ax.plot(ref["day"], ref[var].detach().cpu().numpy(),
                    label="Default WOFOST72 (uncalibrated)", linewidth=1.6)
        if fitted is not None:
            ax.plot(fitted["day"], fitted[var].detach().cpu().numpy(),
                    label="Hybrid (NN stress)", linewidth=1.6, linestyle="--")
        _plot_observed(ax, plot, obs_col)
        ax.set_title(f"{title} — {label}", fontsize=9)
        ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=30, labelsize=7)
axes[0, 0].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


### 11.3 Learned stress factor over the season

`RFTRA(t)` is what the NN actually produces. A value of 1.0 means "no stress",
a value of 0.0 means "complete shutdown of carbon fixation". Reasonable
trajectories should sit between, dipping during the parts of the season where
the model needs to reduce biomass to match observations.


In [ ]:
#@title Plot: learned RFTRA trajectories { display-mode: "form" }
#@markdown §11.3 — daily stress factor over the season.

fig, axes = plt.subplots(1, len(display_plots), figsize=(4 * len(display_plots), 4),
                          squeeze=False)
for col, plot in enumerate(display_plots):
    ax = axes[0, col]
    key = (plot.Year, plot.Location, plot.Plotnumber)
    fitted = test_results.get(key) or train_results.get(key)
    if fitted is not None:
        ax.plot(fitted["day"], fitted["RFTRA"].detach().cpu().numpy(),
                color="tab:orange", linewidth=2)
    ax.axhline(1.0, color="black", linestyle=":", alpha=0.5, label="no stress")
    ax.set_title(f"{plot.Cultivar}@{plot.Location} {plot.Nitrogen}{plot.Irrigation} {plot.Year}", fontsize=9)
    ax.set_ylabel("RFTRA (learned stress factor)")
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=30, labelsize=7)
plt.tight_layout(); plt.show()


### 11.4 Stress profile by cultivar

Per-plot trajectories show us *one* learned curve at a time, but they don't
answer the question we really care about: *did the NN learn that different
cultivars need different stress profiles?* The cultivar one-hot is an input
feature, but nothing forces the NN to use it.

To check, we collapse the time axis from calendar dates to **DVS** (development
stage, 0 = emergence, 1 = anthesis, 2 = maturity), bin `RFTRA` into DVS bins,
and average across all plots of each cultivar (pooled across N, W, year, site).
If the cultivars come out as visibly different curves, the NN used the cultivar
input meaningfully. If they're nearly identical, it didn't — and your one-hot
embedding wasn't worth the dimensions.


In [ ]:
#@title Plot: RFTRA profile by cultivar { display-mode: "form" }
#@markdown §11.4 — mean RFTRA binned by DVS.

DVS_BINS = np.linspace(0.0, 2.0, 21)
DVS_CENTERS = 0.5 * (DVS_BINS[:-1] + DVS_BINS[1:])


def aggregate_rftra_by_dvs(plots, results_lookups, group_key):
    """Bin RFTRA by DVS, averaging across plots that share `group_key(plot)`.

    `results_lookups` is an iterable of dicts to try in order (e.g. test_results
    then train_results) — the first one containing the plot wins.
    Returns {group_value: rftra_means_array_of_len_DVS_CENTERS}.
    """
    groups = sorted({group_key(p) for p in plots})
    sums = {g: np.zeros(len(DVS_CENTERS)) for g in groups}
    counts = {g: np.zeros(len(DVS_CENTERS)) for g in groups}
    for plot in plots:
        key = (plot.Year, plot.Location, plot.Plotnumber)
        r = None
        for d in results_lookups:
            if key in d:
                r = d[key]; break
        if r is None:
            continue
        dvs = r["DVS"].detach().cpu().numpy()
        rftra = r["RFTRA"].detach().cpu().numpy()
        active = (rftra > 1e-6) & (dvs > 0)
        if not active.any():
            continue
        bin_idx = np.clip(np.digitize(dvs[active], DVS_BINS) - 1, 0, len(DVS_CENTERS) - 1)
        g = group_key(plot)
        for b, v in zip(bin_idx, rftra[active]):
            sums[g][b] += v
            counts[g][b] += 1
    out = {}
    for g in groups:
        means = np.full(len(DVS_CENTERS), np.nan)
        nz = counts[g] > 0
        means[nz] = sums[g][nz] / counts[g][nz]
        out[g] = means
    return out


cultivar_rftra_by_dvs = aggregate_rftra_by_dvs(
    train_plots + test_plots,
    [train_results, test_results],
    lambda p: p.Cultivar,
)

fig, ax = plt.subplots(figsize=(11, 5))
for i, c in enumerate(CULTIVARS):
    curve = cultivar_rftra_by_dvs.get(c)
    if curve is not None and not np.isnan(curve).all():
        ax.plot(DVS_CENTERS, curve, marker="o", linewidth=2,
                label=c, color=plt.cm.tab10(i))
ax.axhline(1.0, color="black", linestyle=":", alpha=0.6)
ax.axvline(1.0, color="red", linestyle=":", alpha=0.6, label="DVS=1 (anthesis)")
ax.set_xlabel("DVS (development stage)")
ax.set_ylabel("mean learned RFTRA")
ax.set_title("Seasonal stress profile per cultivar")
ax.set_ylim(0.0, 1.05); ax.grid(alpha=0.3); ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()


### 11.5 Stress profile by nitrogen treatment

Same DVS-binned aggregation as 11.4, but split by **N level** (N0 / N1 / N2)
instead of cultivar — pooled over all cultivars, sites, and W treatments. The
nitrogen treatment is encoded as a scalar input to the NN (N0=0.0, N1=0.3,
N2=1.3), so this plot is the cleanest test of whether the NN actually uses
that input.

Physiologically we'd expect:

- **N0** (no fertiliser) — most N-limited, so largest stress, lowest RFTRA;
- **N1** (~30% advised) — intermediate;
- **N2** (~130% advised) — least N-stressed, RFTRA closest to 1.

If the three curves come out essentially overlapping, the NN didn't
differentiate by N treatment — it absorbed the N-related residual into other
features (cultivar, weather) or just didn't fit it. That's important
information for *what to look at next*: it might mean the N encoding needs
more weight, the data lacks the signal, or the loss isn't sensitive enough to
treatment differences.


In [ ]:
#@title Plot: RFTRA profile by N treatment { display-mode: "form" }
#@markdown §11.5 — mean RFTRA binned by DVS and N level.

N_LEVELS = ["N0", "N1", "N2"]
n_rftra_by_dvs = aggregate_rftra_by_dvs(
    train_plots + test_plots,
    [train_results, test_results],
    lambda p: p.Nitrogen,
)

n_colors = {"N0": "tab:red", "N1": "tab:orange", "N2": "tab:green"}
n_labels = {
    "N0": "N0 (no fertiliser)",
    "N1": "N1 (30% advised)",
    "N2": "N2 (130% advised)",
}

fig, ax = plt.subplots(figsize=(11, 5))
for n in N_LEVELS:
    curve = n_rftra_by_dvs.get(n)
    if curve is not None and not np.isnan(curve).all():
        ax.plot(DVS_CENTERS, curve, marker="o", linewidth=2,
                color=n_colors[n], label=n_labels[n])
ax.axhline(1.0, color="black", linestyle=":", alpha=0.6)
ax.axvline(1.0, color="red", linestyle=":", alpha=0.6, label="DVS=1 (anthesis)")
ax.set_xlabel("DVS (development stage)")
ax.set_ylabel("mean learned RFTRA")
ax.set_title("Seasonal stress profile per nitrogen treatment")
ax.set_ylim(0.0, 1.05); ax.grid(alpha=0.3); ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()

print("Mean RFTRA across the active season (DVS > 0) per N treatment:")
for n in N_LEVELS:
    finite = n_rftra_by_dvs[n][~np.isnan(n_rftra_by_dvs[n])]
    if len(finite):
        print(f"  {n}: mean = {finite.mean():.3f}  min = {finite.min():.3f}  max = {finite.max():.3f}")


> ❓ **Discuss with a neighbour**
>
> Compare the two profile plots
> above. Which axis (cultivar / N treatment) shows the cleaner separation? What
> does that tell you about which input dimension the NN found most informative —
> and how confident should you be that the NN's behaviour reflects *real
> physiology* versus *spurious correlation* in this small dataset?
>


## 12. Reference comparison: default WOFOST

Below we line up the trained hybrid against the *uncalibrated* default
WOFOST72_PP on per-variable normalised RMSE. Before reading the bars, please
keep the framing in mind:

> ⚠️ **This is not an "is hybrid better than WOFOST?" benchmark.**
>
> The default WOFOST72_PP run here uses stock potato parameters with no
> cultivar-specific calibration and a generic sowing schedule for all plots.
> A serious comparison would (i) calibrate per-cultivar genetic parameters
> against the same training data, (ii) match the sowing/management to each
> plot, (iii) ideally use `Wofost72_WLP` so water balance is in play.
>
> What this bar chart *does* show is the **gap that the hybrid is targeting**:
> the difference between the uncalibrated PP trajectory and the observations.
> The hybrid closes most of that gap, so we can see that *the modelling
> approach works* — gradient descent through the engine successfully shapes
> the NN to fit the residuals. The bars are a *diagnostic*, not a leaderboard.

A train/test gap also matters here: small = good generalisation; large =
overfitting on the training plots.


In [ ]:
#@title Plot: hybrid vs WOFOST RMSE { display-mode: "form" }
#@markdown §12 — per-variable normalised RMSE bars.

def per_plot_diag(plots, results_dict):
    out = {k: [] for k in NORMALIZED_WEIGHTS}
    for plot in plots:
        key = (plot.Year, plot.Location, plot.Plotnumber)
        r = results_dict.get(key)
        if r is None:
            continue
        idx, tgt = get_plot_observations(plot, r["day"])
        if idx is None:
            continue
        for name, target in tgt.items():
            pred = r[name].index_select(0, idx)
            valid = torch.isfinite(target)
            if not torch.any(valid):
                continue
            pred = pred[valid]; target_valid = target[valid]
            scale = torch.mean(target_valid).abs().clamp_min(1e-6)
            rmse = torch.sqrt(torch.mean(((pred - target_valid) / scale) ** 2))
            out[name].append(rmse.item())
    return out


train_diag_per_plot = per_plot_diag(train_plots, train_results)
test_diag_per_plot = per_plot_diag(test_plots, test_results)
ref_diag_per_plot = per_plot_diag(train_plots + test_plots, REFERENCE_PLOT_RESULTS)

variables = list(NORMALIZED_WEIGHTS.keys())
n_vars = len(variables)
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(n_vars)
width = 0.27

train_means = [np.mean(train_diag_per_plot[v]) if train_diag_per_plot[v] else 0 for v in variables]
test_means = [np.mean(test_diag_per_plot[v]) if test_diag_per_plot[v] else 0 for v in variables]
ref_means = [np.mean(ref_diag_per_plot[v]) if ref_diag_per_plot[v] else 0 for v in variables]

ax.bar(x - width, ref_means, width, label="Default WOFOST72 (uncalibrated)",
       color="lightgray", edgecolor="black")
ax.bar(x, train_means, width, label="Hybrid, on TRAIN", color="steelblue", edgecolor="black")
ax.bar(x + width, test_means, width, label="Hybrid, on TEST", color="orange", edgecolor="black")

ax.set_xticks(x); ax.set_xticklabels(variables)
ax.set_ylabel("normalised RMSE (averaged over plots)")
ax.set_title("Per-variable RMSE: uncalibrated WOFOST reference vs. trained hybrid")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

print()
print("Per-variable summary:")
for v, ref, tr, te in zip(variables, ref_means, train_means, test_means):
    print(f"  {v:<6} default={ref:.3f}  train={tr:.3f}  test={te:.3f}  "
          f"(test/train: {te / max(tr, 1e-6):.2f})")


> ❓ **Discuss with a neighbour**
>
> Which variables does the hybrid
> close the gap on most? Which does it barely move? Where do you think the
> remaining test-set error comes from — the NN's capacity, the feature set, or
> genuinely uncalibrated parts of WOFOST that the NN can't reach?
>


## 13. Comparison vs. a pure-ML LSTM

To complete the picture we now train a **pure data-driven model with no physics
in the loop**: a small LSTM that maps per-day features → daily organ biomass.
This tells us how much of the hybrid's fit comes from the *engine* versus the
*data*.

We choose an LSTM (not just an MLP) because the hybrid model implicitly gets
weather *history* through the engine's state variables (`LAI`, `WLV`, `DVS`,
…). To make the pure-ML reference comparable, it needs a way to integrate past
weather too — that's what the LSTM's hidden state does.

**Architecture:**

```
Per-day input (14):
  - Weather:    VPD, TMAX, 7-day rolling rain, IRRAD               [4]
  - Time:       days_since_sowing / 200                            [1]
  - Treatment:  site, N level, W level, cultivar one-hot           [9]

       inputs (200, 14)
              │
              ▼
       LSTM (32 hidden, 1 layer)
              │
              ▼
       Linear → Sigmoid × output_scales
              │
              ▼
       4 daily organ predictions   (WLV, TWST, TWSO, LAI)
```

The LSTM has access to the *same* inputs as the hybrid — same weather, same
treatment context. The only difference is: no physics in the loop. So any
performance gap between the two is attributable to the inductive bias of the
WOFOST engine.

> ⚠️ **Same caveat as the WOFOST baseline.** This LSTM is a vanilla reference
> architecture, not a tuned state-of-the-art. The point of the comparison is
> *qualitative*: how stable / how data-hungry / how prone to overfitting is a
> black-box ML model versus a hybrid one on this size of dataset? Don't read
> these bars as "hybrid > LSTM in absolute terms" — read them as "physics
> contributes inductive bias that matters when you have ~140 training plots."


### 13.1 LSTM data preparation

Pre-build the per-day feature sequences (one per plot) and the
observation-index / target tensors. Doing this once up front means the training
loop just slices into prebuilt tensors.


In [ ]:
#@title LSTM data preparation { display-mode: "form" }
#@markdown Pre-build per-plot feature sequences and observation targets for the pure-ML baseline.

# Output scales: roughly the order-of-magnitude max we'd expect per organ.
# The sigmoid output is multiplied by these so the network outputs in the right
# range without needing the LSTM to produce huge raw logits.
PURE_OUTPUT_SCALES = torch.tensor(
    [3000.0, 2500.0, 20000.0, 8.0],   # WLV, TWST, TWSO, LAI
    dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
)

# Sowing dates — same for all plots in this dataset.
SOWING_DATE_BY_YEAR = {
    2019: pd.Timestamp("2019-04-20"),
    2020: pd.Timestamp("2020-04-20"),
}
DAYS_NORMALIZER = 200.0

PURE_OBS_VARS    = ["LeavesDW", "StemDW", "tubersDW", "LAI"]
PURE_OBS_TO_PCSE = ["WLV", "TWST", "TWSO", "LAI"]
PURE_N_FEATURES  = WEATHER_FEATURE_DIM + 1 + PLOT_CONTEXT_DIM


def _raw_pure_features(plot, date):
    norm_date = pd.Timestamp(date).normalize()
    wf = WEATHER_FEATURES[plot.Location].get(norm_date)
    if wf is None:
        wf = torch.zeros(WEATHER_FEATURE_DIM, dtype=ComputeConfig.get_dtype(),
                         device=ComputeConfig.get_device())
    days = (norm_date - SOWING_DATE_BY_YEAR[plot.Year]).days
    days_t = torch.tensor([days / DAYS_NORMALIZER], dtype=ComputeConfig.get_dtype(),
                          device=ComputeConfig.get_device())
    ctx = make_plot_context_tensor(plot.Cultivar, plot.Nitrogen, plot.Irrigation, plot.Location)
    return torch.cat([wf, days_t, ctx])


def pure_features_for_date(plot, date):
    raw = _raw_pure_features(plot, date)
    if "PURE_FEATURES_MEAN" in globals() and "PURE_FEATURES_STD" in globals():
        return (raw - PURE_FEATURES_MEAN) / PURE_FEATURES_STD
    return raw


# Training-set normalisation stats — computed from per-observation features only,
# so the LSTM input is roughly zero-mean unit-variance on the train set.
def _build_raw(plots):
    feats = []
    for plot in plots:
        rows = obs_df[
            (obs_df["Year"] == plot.Year)
            & (obs_df["Location"] == plot.Location)
            & (obs_df["Plotnumber"] == plot.Plotnumber)
            & obs_df[PURE_OBS_VARS].notna().any(axis=1)
        ]
        for _, r in rows.iterrows():
            feats.append(_raw_pure_features(plot, r["Date"]))
    return torch.stack(feats) if feats else None


_raw_train = _build_raw(train_plots)
PURE_FEATURES_MEAN = _raw_train.mean(dim=0)
PURE_FEATURES_STD  = _raw_train.std(dim=0).clamp_min(1e-3)
# Keep one-hots / binary flags in their raw {0, 1} scale.
PURE_FEATURES_STD[WEATHER_FEATURE_DIM + 1:]  = 1.0
PURE_FEATURES_MEAN[WEATHER_FEATURE_DIM + 1:] = 0.0

# Re-use the same per-variable loss weights as the hybrid model so the two
# losses are directly comparable.
PURE_VARIABLE_WEIGHTS = torch.tensor(
    [NORMALIZED_WEIGHTS[v] for v in PURE_OBS_TO_PCSE],
    dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
)


LSTM_SEQ_LEN = 200    # days from sowing
LSTM_HIDDEN  = 32


class PureLSTM(torch.nn.Module):
    """Sequence-aware pure-ML reference. (seq_len, 14) → (seq_len, 4)."""

    def __init__(self, n_features=14, hidden=LSTM_HIDDEN, n_outputs=4, output_scales=None):
        super().__init__()
        self.lstm = torch.nn.LSTM(
            input_size=n_features, hidden_size=hidden,
            num_layers=1, batch_first=True,
        )
        self.head = torch.nn.Linear(hidden, n_outputs)
        if output_scales is None:
            output_scales = PURE_OUTPUT_SCALES
        self.register_buffer("output_scales", output_scales.clone().to(
            dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
        ))
        self.to(device=ComputeConfig.get_device(), dtype=ComputeConfig.get_dtype())

    def forward(self, seq):
        if seq.dim() == 2:
            seq = seq.unsqueeze(0)
        h, _ = self.lstm(seq)              # (B, T, hidden)
        raw = self.head(h)                 # (B, T, 4)
        return torch.sigmoid(raw) * self.output_scales


def build_lstm_data(plot, n_days=LSTM_SEQ_LEN):
    sowing = SOWING_DATE_BY_YEAR[plot.Year]
    feats = []
    for d in range(n_days):
        date = sowing + pd.Timedelta(days=d)
        feats.append(pure_features_for_date(plot, date))
    seq = torch.stack(feats)

    rows = obs_df[
        (obs_df["Year"] == plot.Year)
        & (obs_df["Location"] == plot.Location)
        & (obs_df["Plotnumber"] == plot.Plotnumber)
        & obs_df[PURE_OBS_VARS].notna().any(axis=1)
    ].sort_values("Date").reset_index(drop=True)
    if rows.empty:
        return seq, None, None
    indices, targets = [], []
    for _, r in rows.iterrows():
        d = (pd.Timestamp(r["Date"]).normalize() - sowing).days
        d = max(0, min(d, n_days - 1))
        indices.append(d)
        targets.append([r[v] for v in PURE_OBS_VARS])
    idx_t = torch.tensor(indices, dtype=torch.long, device=ComputeConfig.get_device())
    tgt_t = torch.tensor(targets, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    return seq, idx_t, tgt_t


print("Pre-building LSTM sequences for every plot...")
lstm_data_train = {p: build_lstm_data(p) for p in train_plots}
lstm_data_test  = {p: build_lstm_data(p) for p in test_plots}
print(f"  train sequences: {len(lstm_data_train)} plots, each ({LSTM_SEQ_LEN}, {PURE_N_FEATURES})")
print(f"  test  sequences: {len(lstm_data_test)} plots, each ({LSTM_SEQ_LEN}, {PURE_N_FEATURES})")


### 13.2 Train (or load) the LSTM

The LSTM trains the same way as the hybrid — pooled normalised RMSE, Adam,
early stopping. To prevent overfitting on the small dataset, we (a) carve a
separate validation split out of the training plots so early stopping does not
peek at the test set, and (b) add a touch of weight decay. As with the hybrid,
we load a pre-trained checkpoint by default.


In [ ]:
#@title Train LSTM { display-mode: "form" }
lstm_model_path = MODEL_DIR / "pure_lstm_random.pt"


def lstm_pooled_loss(lstm, plot_data_dict, weights_tensor):
    var_total = torch.zeros(4, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    var_count = torch.zeros(4, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    plot_preds = {}
    for plot, (seq, idx, tgt) in plot_data_dict.items():
        if idx is None or len(idx) == 0:
            continue
        pred_full = lstm(seq)
        pred_at_obs = pred_full[0, idx, :]
        plot_preds[plot] = pred_full[0]
        for i in range(4):
            valid = torch.isfinite(tgt[:, i])
            if not valid.any():
                continue
            pv = pred_at_obs[valid, i]
            tv = tgt[valid, i]
            scale = tv.mean().abs().clamp_min(1e-6)
            rmse = torch.sqrt(torch.mean(((pv - tv) / scale) ** 2))
            var_total[i] = var_total[i] + weights_tensor[i] * rmse
            var_count[i] = var_count[i] + 1
    n_plots = sum(1 for _, (_, idx, _) in plot_data_dict.items() if idx is not None)
    if n_plots == 0:
        return torch.zeros((), dtype=ComputeConfig.get_dtype()), {}, {}
    total = var_total.sum() / n_plots
    diag = {
        name: (var_total[i] / var_count[i].clamp_min(1)).item() / weights_tensor[i].item()
        for i, name in enumerate(PURE_OBS_TO_PCSE)
    }
    return total, diag, plot_preds


torch.manual_seed(23)
pure_lstm = PureLSTM(n_features=PURE_N_FEATURES, hidden=LSTM_HIDDEN)
lstm_n_params = sum(p.numel() for p in pure_lstm.parameters())
print(f"PureLSTM: {lstm_n_params} parameters")

if lstm_model_path.exists() and not FORCE_RETRAIN:
    saved = torch.load(lstm_model_path, weights_only=False)
    pure_lstm.load_state_dict(saved["state_dict"])
    lstm_run = saved["lstm_run"]
    print(f"Loaded saved LSTM from {lstm_model_path.name}")
    print(f"  Saved train loss: {lstm_run['train_history'][-1]:.4f}")
    print(f"  Saved test  loss: {lstm_run['test_history'][-1]:.4f}")
else:
    LSTM_VAL_FRACTION = 0.2
    LSTM_VAL_SEED = 1234
    _val_rng = np.random.default_rng(LSTM_VAL_SEED)
    _all_train_plots_for_lstm = list(lstm_data_train.keys())
    _shuffled = list(_all_train_plots_for_lstm)
    _val_rng.shuffle(_shuffled)
    _n_val = max(1, int(round(len(_shuffled) * LSTM_VAL_FRACTION)))
    _val_plots = _shuffled[:_n_val]
    _train_subset_plots = _shuffled[_n_val:]
    lstm_data_train_subset = {p: lstm_data_train[p] for p in _train_subset_plots}
    lstm_data_val = {p: lstm_data_train[p] for p in _val_plots}
    print(f"  LSTM split: train_subset={len(lstm_data_train_subset)}  "
          f"val={len(lstm_data_val)}  test={len(lstm_data_test)}")

    optimizer = torch.optim.Adam(pure_lstm.parameters(), lr=0.005, weight_decay=1e-3)
    lstm_run = {"train_history": [], "val_history": [], "test_history": [], "diag_history": []}

    LSTM_MAX_STEPS, LSTM_PATIENCE, LSTM_MIN_DELTA = 1500, 100, 1e-3
    best_val_loss, best_step = float("inf"), -1
    best_state = copy.deepcopy(pure_lstm.state_dict())

    for step in range(LSTM_MAX_STEPS):
        optimizer.zero_grad()
        train_loss, train_diag, _ = lstm_pooled_loss(pure_lstm, lstm_data_train_subset, PURE_VARIABLE_WEIGHTS)
        with torch.no_grad():
            val_loss, _, _ = lstm_pooled_loss(pure_lstm, lstm_data_val, PURE_VARIABLE_WEIGHTS)
            test_loss, _, _ = lstm_pooled_loss(pure_lstm, lstm_data_test, PURE_VARIABLE_WEIGHTS)
        lstm_run["train_history"].append(train_loss.item())
        lstm_run["val_history"].append(val_loss.item())
        lstm_run["test_history"].append(test_loss.item())
        lstm_run["diag_history"].append(train_diag)

        if step % 50 == 0:
            marker = " *" if val_loss.item() < best_val_loss - LSTM_MIN_DELTA else ""
            print(f"  pure_lstm step {step:04d} | train={train_loss.item():.4f} "
                  f"val={val_loss.item():.4f}{marker} test={test_loss.item():.4f}")

        if val_loss.item() < best_val_loss - LSTM_MIN_DELTA:
            best_val_loss = val_loss.item(); best_step = step
            best_state = copy.deepcopy(pure_lstm.state_dict())

        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(pure_lstm.parameters(), max_norm=2.0)
        optimizer.step()

        if step - best_step >= LSTM_PATIENCE:
            print(f"  pure_lstm early stopping at step {step} (best val {best_val_loss:.4f})")
            break

    pure_lstm.load_state_dict(best_state)
    torch.save({
        "state_dict": pure_lstm.state_dict(),
        "lstm_run": lstm_run,
        "n_features": PURE_N_FEATURES,
        "seq_len": LSTM_SEQ_LEN,
        "hidden": LSTM_HIDDEN,
    }, lstm_model_path)
    print(f"Saved trained LSTM to {lstm_model_path}")

with torch.no_grad():
    final_lstm_train_loss, final_lstm_train_diag, lstm_train_preds = lstm_pooled_loss(
        pure_lstm, lstm_data_train, PURE_VARIABLE_WEIGHTS,
    )
    final_lstm_test_loss, final_lstm_test_diag, lstm_test_preds = lstm_pooled_loss(
        pure_lstm, lstm_data_test, PURE_VARIABLE_WEIGHTS,
    )

print()
print(f"Pure LSTM | final train loss = {final_lstm_train_loss.item():.4f}")
print(f"Pure LSTM | final test  loss = {final_lstm_test_loss.item():.4f}")


### 13.3 Three-way comparison

Three views, each telling a different story.

**(i) Per-variable test RMSE.** Default WOFOST (uncalibrated reference),
hybrid, and pure LSTM side by side on the held-out test set. The hybrid
typically lands in between — it doesn't have the LSTM's raw memorisation
capacity but it generalises more reliably (see view (iii)).


In [ ]:
#@title Plot: three-model test RMSE { display-mode: "form" }
#@markdown §13 — per-variable bars on the test set.

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(PURE_OBS_TO_PCSE))
width = 0.27
ax.bar(x - width, [ref_diag[v] for v in PURE_OBS_TO_PCSE], width,
       label="Default WOFOST72 (uncalibrated)", color="lightgray", edgecolor="black")
ax.bar(x, [final_test_diag.get(v, np.nan) for v in PURE_OBS_TO_PCSE], width,
       label="Hybrid (PP + NN stress)", color="tab:orange", edgecolor="black")
ax.bar(x + width, [final_lstm_test_diag.get(v, np.nan) for v in PURE_OBS_TO_PCSE], width,
       label="Pure LSTM (no physics)", color="tab:purple", edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(PURE_OBS_TO_PCSE)
ax.set_ylabel("normalised RMSE on TEST set")
ax.set_title("Per-variable test RMSE: three models, random 80/20 split")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()


**(ii) Trajectories on a single plot.** Showing all three models on the
*same* sample plot makes the qualitative differences clear: the hybrid's
trajectories obey the engine's physics (smooth, physiologically plausible
curves with the right shape), while the LSTM produces whatever shape minimises
its loss — sometimes biologically odd kinks or wiggles, particularly near the
start of the season where it has little context.


In [ ]:
#@title Plot: three-model trajectories { display-mode: "form" }
#@markdown §13 — one example plot, all three models.

sample_plot_for_lstm = test_plots[0] if test_plots else train_plots[0]

# Build LSTM trajectory for this plot
sample_data = lstm_data_test.get(sample_plot_for_lstm) or lstm_data_train.get(sample_plot_for_lstm)
if sample_data is None:
    seq_sample, _, _ = build_lstm_data(sample_plot_for_lstm)
else:
    seq_sample = sample_data[0]
with torch.no_grad():
    lstm_traj = pure_lstm(seq_sample)[0].cpu().numpy()
sowing_for_sample = SOWING_DATE_BY_YEAR[sample_plot_for_lstm.Year]
lstm_dates = [sowing_for_sample + pd.Timedelta(days=d) for d in range(LSTM_SEQ_LEN)]

ref_traj_sample = REFERENCE_PLOT_RESULTS.get(
    (sample_plot_for_lstm.Year, sample_plot_for_lstm.Location, sample_plot_for_lstm.Plotnumber)
)
hybrid_traj_sample = (
    test_results.get((sample_plot_for_lstm.Year, sample_plot_for_lstm.Location, sample_plot_for_lstm.Plotnumber))
    or train_results.get((sample_plot_for_lstm.Year, sample_plot_for_lstm.Location, sample_plot_for_lstm.Plotnumber))
)
obs_for_lstm = obs_df[
    (obs_df["Year"] == sample_plot_for_lstm.Year)
    & (obs_df["Location"] == sample_plot_for_lstm.Location)
    & (obs_df["Plotnumber"] == sample_plot_for_lstm.Plotnumber)
]

ORGAN_LIST_LSTM = [
    ("WLV",  "LeavesDW",  "Leaves (WLV)", 0),
    ("TWST", "StemDW",    "Stems (TWST)", 1),
    ("TWSO", "tubersDW",  "Storage organs (TWSO)", 2),
    ("LAI",  "LAI",       "LAI", 3),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, (var, obs_col, title, lstm_idx) in zip(axes.ravel(), ORGAN_LIST_LSTM):
    if ref_traj_sample is not None and var in ref_traj_sample:
        ax.plot(ref_traj_sample["day"], ref_traj_sample[var].detach().cpu().numpy(),
                label="Default WOFOST72 (uncalibrated)", linewidth=2, color="tab:blue")
    if hybrid_traj_sample is not None and var in hybrid_traj_sample:
        ax.plot(hybrid_traj_sample["day"], hybrid_traj_sample[var].detach().cpu().numpy(),
                label="Hybrid (PP + NN stress)", linewidth=2, linestyle="--", color="tab:orange")
    ax.plot(lstm_dates, lstm_traj[:, lstm_idx],
            label="Pure LSTM (no physics)", linewidth=2, linestyle=":", color="tab:purple")
    if obs_col in obs_for_lstm.columns:
        finite_obs = obs_for_lstm[obs_for_lstm[obs_col].notna()]
        if len(finite_obs):
            ax.scatter(finite_obs["Date"], finite_obs[obs_col],
                       s=44, color="black", zorder=5, label="Observed")
    ax.set_title(title); ax.set_ylabel("kg/ha (LAI: m²/m²)")
    ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=30)
axes[0, 0].legend(loc="upper left", fontsize=9)
plot_label = (f"WOFOST + hybrid + LSTM — {sample_plot_for_lstm.Cultivar}@{sample_plot_for_lstm.Location} "
              f"{sample_plot_for_lstm.Nitrogen}{sample_plot_for_lstm.Irrigation} {sample_plot_for_lstm.Year}")
fig.suptitle(plot_label, y=1.02, fontsize=13)
plt.tight_layout(); plt.show()


**(iii) Train vs. test gap — the headline result.** This is where the
inductive bias of the engine pays off. The LSTM tends to fit the training set
*very* well (it has the capacity to memorise the ~140 training plots) but its
test loss is often markedly higher — a sign of overfitting. The hybrid has a
much smaller train/test ratio because the physics in the loop constrains what
shapes its predictions can take in the first place.

If you re-train the LSTM with a different seed, you'll usually see a different
test-set loss; re-training the hybrid is far more reproducible. That
**stability / consistency advantage** is one of the main reasons to keep
physics in the loop on small datasets.


In [ ]:
#@title Plot: hybrid vs LSTM generalisation { display-mode: "form" }
#@markdown §13 — pooled train/test loss comparison.

hybrid_train = final_train_loss.item()
hybrid_test  = final_test_loss.item()
lstm_train_v = final_lstm_train_loss.item()
lstm_test_v  = final_lstm_test_loss.item()

print(f"Generalisation gap (test/train) on random 80/20 split:")
print(f"  Hybrid    : train={hybrid_train:.4f}  test={hybrid_test:.4f}  "
      f"ratio={hybrid_test/max(hybrid_train,1e-9):.2f}")
print(f"  Pure LSTM : train={lstm_train_v:.4f}  test={lstm_test_v:.4f}  "
      f"ratio={lstm_test_v/max(lstm_train_v,1e-9):.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
models = ["Hybrid\n(PP + NN stress)", "Pure LSTM\n(no physics)"]
trains = [hybrid_train, lstm_train_v]
tests  = [hybrid_test, lstm_test_v]
xpos = np.arange(len(models))
ax.bar(xpos - 0.2, trains, 0.4, label="train loss", color="tab:blue", edgecolor="black")
ax.bar(xpos + 0.2, tests,  0.4, label="test loss",  color="tab:orange", edgecolor="black")
ax.set_xticks(xpos); ax.set_xticklabels(models)
ax.set_ylabel("pooled normalised RMSE")
ax.set_title("Train vs. test loss — physics keeps the hybrid honest")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()


> ❓ **Discuss with a neighbour**
>
> Look at the three views above.
> On *training* data, which model wins? On *test* data? And if we would do spits by cultivar which would you expect to degrade more? Why?
>


## 14. Bonus: differentiability for free

This is the bit that makes `diffwofost` *different* from a pure forward
simulator. Because every operation in the engine is implemented in PyTorch, we
can compute gradients of *any* simulated quantity with respect to *any*
parameter — using one forward pass and one backward pass.

To demonstrate, we ask: *how much does the final tuber yield (`TWSO`) change if
each of `SPAN`, `TSUM1`, or `TSUM2` were perturbed by one unit?*

In a non-differentiable simulator, you'd run hundreds of finite-difference
simulations. With autograd, you get all sensitivities at once.


In [ ]:
sample_plot = test_plots[0] if test_plots else train_plots[0]
print(f"Sample plot: {sample_plot.Cultivar}@{sample_plot.Location} "
      f"{sample_plot.Nitrogen}{sample_plot.Irrigation} {sample_plot.Year}")


def run_with_param_overrides(plot, nn, **scalar_overrides):
    fb = PlotFeatureBuilder(WEATHER_FEATURES[plot.Location],
                            make_plot_context_tensor(plot.Cultivar, plot.Nitrogen,
                                                     plot.Irrigation, plot.Location))
    cfg = build_config(NNStressFactor, et_kwargs={"nn_model": nn, "feature_builder": fb})
    pp = copy.deepcopy(parameter_provider)
    for name, value in scalar_overrides.items():
        pp.set_override(name, value, check=False)
    engine = Engine(config=cfg)
    engine.setup(pp, WEATHER_DATA_PROVIDERS[plot.Location],
                 YAMLAgroManagementReader(agro_paths[plot.Year]))
    engine.run_till_terminate()
    return results_to_tensors(engine.get_output())


span_t = torch.tensor(35.0, dtype=ComputeConfig.get_dtype(), requires_grad=True)
tsum1_t = torch.tensor(150.0, dtype=ComputeConfig.get_dtype(), requires_grad=True)
tsum2_t = torch.tensor(1550.0, dtype=ComputeConfig.get_dtype(), requires_grad=True)

results_grad = run_with_param_overrides(
    sample_plot, stress_nn,
    SPAN=span_t, TSUM1=tsum1_t, TSUM2=tsum2_t,
)
twso_final = results_grad["TWSO"][-1]
print(f"\nSimulated final TWSO = {twso_final.item():.1f} kg/ha")

# ONE backward pass yields gradients to ALL parameters at once
twso_final.backward()
print()
print("Gradients of final TWSO with respect to each parameter (autograd):")
print(f"  d(TWSO)/d(SPAN)  = {span_t.grad.item():+.2f} kg/ha per day")
print(f"  d(TWSO)/d(TSUM1) = {tsum1_t.grad.item():+.4f} kg/ha per (degC*day)")
print(f"  d(TWSO)/d(TSUM2) = {tsum2_t.grad.item():+.4f} kg/ha per (degC*day)")


Translated into "% change in final TWSO per typical-magnitude
perturbation", the sensitivities look like this:


In [ ]:
#@title Plot: yield parameter sensitivities { display-mode: "form" }
#@markdown §14 — % change in final TWSO per perturbation.

TYPICAL_PERTURB = {"SPAN": 5.0, "TSUM1": 50.0, "TSUM2": 100.0}
gradients = {
    "SPAN":  span_t.grad.item(),
    "TSUM1": tsum1_t.grad.item(),
    "TSUM2": tsum2_t.grad.item(),
}
twso_baseline = twso_final.item()
sensitivities_pct = {
    name: (gradients[name] * TYPICAL_PERTURB[name]) / twso_baseline * 100
    for name in gradients
}

fig, ax = plt.subplots(figsize=(10, 4))
names = list(sensitivities_pct.keys())[::-1]
values = [sensitivities_pct[n] for n in names]
colors = ["tab:blue" if v >= 0 else "tab:red" for v in values]
bars = ax.barh(names, values, color=colors, edgecolor="black", height=0.6)

for bar, n, v in zip(bars, names, values):
    label = f"{v:+.1f}%   (perturbing {n} by {TYPICAL_PERTURB[n]:g} units)"
    if v >= 0:
        ax.text(max(v, 0.05) + 0.05, bar.get_y() + bar.get_height() / 2, label,
                va="center", ha="left", fontsize=10)
    else:
        ax.text(min(v, -0.05) - 0.05, bar.get_y() + bar.get_height() / 2, label,
                va="center", ha="right", fontsize=10)

ax.axvline(0, color="black", linewidth=0.7)
xlim = max(abs(v) for v in values) * 1.6 + 0.5
ax.set_xlim(-xlim, xlim)
ax.set_xlabel("% change in final TWSO per typical-magnitude perturbation")
ax.set_title(f"Yield sensitivity (baseline TWSO = {twso_baseline:.0f} kg/ha)")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.show()


**Why this matters.**

- **Targeted calibration.** The gradient tells you *which* parameter to tune
  first when residuals appear.
- **Sensitivity analysis at scale.** Same cost per plot, regardless of how
  many parameters you probe.
- **Optimisation through the engine.** "What sowing date maximises yield under
  these climate scenarios?" is now a smooth optimisation problem.
- **Variational data assimilation.** Gradient-based DA (e.g. 4D-Var) uses
  exactly this computational structure.

The hybrid stress NN is one application of `diffwofost`. Gradients are the
underlying capability.


## 15. Recap and what's next

We just walked through the full pipeline of a hybrid crop model:

1. **Loaded a real field-trial dataset** (174 plot-years, two sites, six
   cultivars, three N levels, two W levels).
2. **Diagnosed the gap**: default (uncalibrated) WOFOST72_PP overpredicts
   biomass on water/N-stressed plots because it has no stress module.
3. **Replaced the stress component** with a tiny NN producing daily `RFTRA`.
4. **Trained the NN end-to-end** with gradient descent through the WOFOST
   engine.
5. **Inspected what the NN learned**: per-plot trajectories, per-cultivar and
   per-N-treatment stress profiles — checking that the NN actually uses its
   inputs.
6. **Compared against two reference points**: an uncalibrated WOFOST baseline
   (to see what the NN is correcting) and a pure-ML LSTM (to see what the
   engine's inductive bias contributes — especially the smaller train/test
   gap).
7. **Saw the bonus**: free gradients with respect to any parameter.

### Mental model: where does each model belong?

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **Calibrated WOFOST** | Interpretable, transferable, mechanistic | Labour-intensive to fit; assumptions break |
| **Uncalibrated WOFOST (this notebook's reference)** | None — just a starting point | Systematic biases, especially under stress |
| **Pure LSTM** | High fit on training data | Unstable, brittle on held-out conditions |
| **Hybrid (this tutorial)** | Reliable shape from physics, NN closes the gap | NN is opaque; needs careful feature design |

### Where to go from here

- **Partitioning hybrid**:
  https://github.com/WUR-AI/diffWOFOST/blob/main/docs/notebooks/hybrid_partitioning_wofost72.ipynb
  applies the same idea to a *different* WOFOST component (carbon
  partitioning), useful for understanding what hybrid replacement can and
  can't do.
- **Pure calibration**: see https://github.com/WUR-AI/diffWOFOST/blob/main/docs/notebooks/optimization_phenology.ipynb (and friends) for parameter-only
  calibration (no NN) using the same differentiable engine — that's the
  apples-to-apples WOFOST comparison this notebook deliberately does not do.

### Caveats

- The trained NN is specific to this sowing schedule and weather window.
  Predictions outside the training distribution are unreliable.
- Cultivar-specific genetics (`SPAN`, `TSUM2`) are *not* calibrated here.
  Residuals related to leaf longevity and time-to-maturity remain visible.

That's it — happy hybrid modelling!
